# weight-decay-l2-add — worked example 3: L2-into-grad vs decoupled AdamW

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `weight-decay-l2-add`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Classic L2 folds decay into the gradient, so it passes through Adam's momentum and variance buffers. AdamW instead decouples decay, subtracting `lr * lmda * theta` directly from the parameter after the Adam update. Because the variance normalization touches the L2 term but not the AdamW term, the two trajectories diverge after even a couple of steps.

## Worked solution

We run two Adam-style trajectories from the same start and show they differ.

1. Both paths start at `theta0` with `m = v = 0` and use the same two gradients.
2. L2 path: each step folds `lmda * theta` into `g`, then runs the standard bias-corrected Adam update. The decay therefore flows through `m`, `v`, and the `1/sqrt(v_hat)` scaling.
3. AdamW path: Adam runs on the raw gradient, and decay is applied as a separate `- lr * lmda * theta` term outside the variance normalization.
4. After two steps the final parameters differ — the whole point of the AdamW correction.

We print both final tensors and confirm they are not equal.

In [ ]:
import torch as t

def two_step_l2_vs_adamw(theta0, grads, lr, beta1, beta2, eps, lmda):
    def run(decoupled):
        theta = theta0.clone()
        m = t.zeros_like(theta)
        v = t.zeros_like(theta)
        for step in (1, 2):
            g = grads[step - 1] if decoupled else grads[step - 1] + lmda * theta
            m = beta1 * m + (1 - beta1) * g
            v = beta2 * v + (1 - beta2) * g * g
            m_hat = m / (1 - beta1 ** step)
            v_hat = v / (1 - beta2 ** step)
            theta = theta - lr * (m_hat / (v_hat.sqrt() + eps))
            if decoupled:
                theta = theta - lr * lmda * theta
        return theta
    return run(False), run(True)

theta0 = t.tensor([1.0, -1.0, 0.5])
grads = [t.tensor([0.2, -0.1, 0.3]), t.tensor([0.1, 0.0, -0.2])]
l2, adamw = two_step_l2_vs_adamw(theta0, grads, lr=0.1, beta1=0.9, beta2=0.999, eps=1e-8, lmda=0.1)
print('L2:   ', [round(x, 5) for x in l2.tolist()])
print('AdamW:', [round(x, 5) for x in adamw.tolist()])
print('diverged:', not t.allclose(l2, adamw))